# Project KIRA: ADV-003 Adaptive Defense Curve & Anti-Forgetting Evaluation

**Experiment ID**: `ADV-003`  
**Objective**: Closed-loop adversarial defense curve across 10 sequential rounds and 3 control arms (`static_blue`, `adaptive_challenger`, `replay_control`).  
**Compute Mode**: CPU Only (Standard Kaggle CPU Kernel)  
**Authoritative Baseline**: `run_tiny_s20260827_193f7897_40997ab` (Read-only substrate)  
**Deadline Target**: < 30 minutes wall-clock ceiling (Projected: ~3-5 minutes)


In [ ]:
# 1. Setup Environment, Clone Project-KIRA Repository, and Install Dependencies
import os
import sys
import subprocess
import hashlib
from pathlib import Path

if not os.path.exists("src"):
    if os.path.exists("/kaggle/working/Project-KIRA/.git"):
        os.chdir("/kaggle/working/Project-KIRA")
        print("Pulling latest code from GitHub main...")
        subprocess.run(["git", "fetch", "origin", "main"], check=False)
        subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)
    elif os.path.exists("/kaggle/input/project-kira/src"):
        os.chdir("/kaggle/input/project-kira")
    else:
        print("Cloning latest Project-KIRA repository (--depth 1)...")
        subprocess.run(["rm", "-rf", "/kaggle/working/Project-KIRA"], check=False)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ankit-choubey/Project-KIRA.git", "/kaggle/working/Project-KIRA"], check=True)
        os.chdir("/kaggle/working/Project-KIRA")

if "src" not in sys.path:
    sys.path.insert(0, "src")

!pip install -q polars scikit-learn lightgbm pytest
print("Environment initialized in:", os.getcwd())

In [ ]:
# 2. Pre-flight: Verify Baseline 22/22 Artifacts & ADV-001 Memory Integrity
!python3 tools/audit_authoritative_run.py run_tiny_s20260827_193f7897_40997ab

adv001_path = Path("research_runs/ADVANCED/ADV-001/attack_memory.jsonl")
if adv001_path.exists():
    h = hashlib.sha256()
    with open(adv001_path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    print(f"ADV-001 Attack Memory SHA-256: {h.hexdigest()}")
    assert h.hexdigest() == "7f37b59333a82d8fd04ab4e3582435cb798fa6e97cbcd1dc248122967e8087f7", "ADV-001 hash mismatch!"
    print("PRE-FLIGHT ADV-001 INTEGRITY: PASS")

In [ ]:
# 3. Execute Complete 3-Arm ADV-003 Large Adaptive Defense Curve Experiment
# Arm A (Static Blue) + Arm B (Adaptive Challenger) + Arm C (Replay Control)
!PYTHONPATH=src python3 -m mcdl.research.advanced.adv003.runner \
    --scale large \
    --output-dir research_runs/ADVANCED/ADV-003

In [ ]:
# 4. Post-Execution Regression Suite & Baseline Audit
!PYTHONPATH=src pytest tests/unit/research/test_adv003.py tests/unit/research/test_adv002.py tests/unit/research/test_adv001.py -v
!python3 tools/audit_authoritative_run.py run_tiny_s20260827_193f7897_40997ab

In [ ]:
# 5. Package Evidence for Download
!zip -r /kaggle/working/adv003_evidence.zip research_runs/ADVANCED/ADV-003/
print("Evidence package created: /kaggle/working/adv003_evidence.zip")